# GEMS 2.0 — Master pipeline

**One notebook for the whole workflow:** environment → data acquisition → validation → preprocessing → geological EDA repair → spatial splits → feature store → baselines → research experiments → discovery evaluation → ensembling → full inference → submission → reports.

Every stage **checks Drive first, reuses valid outputs, builds only what is missing, saves to Google Drive immediately and records provenance**. Re-running any cell after a disconnect resumes where it stopped.

```
bootstrap → raw_data → native_grid ─┬─ geological_eda → segment_attributes ─┐
                                    ├─ spatial_folds ───────────────────────┼─→ feature_store ─→ baselines ─→ discovery_validation ─→ ensemble
                                    ├─ discovery_splits ────────────────────┘        │               └─→ advanced_models ──┘
                                    └─ dem_inventory ─→ dem_pilot (needs feature_store)   └─→ full_inference ─→ submission
                                                                                     reports (always last)
```

| decision | meaning |
|---|---|
| `SKIP_CACHED` | outputs exist, pass content checks, and were made under this configuration (or are pre-existing artifacts, adopted with provenance) |
| `BUILD` / `RESUME` | outputs missing / partial: built, or resumed from completed units and checkpoints |
| `INCOMPATIBLE` | outputs exist but differ from the configuration: **never overwritten**; changed protocols get a new versioned folder |
| `BLOCKED` | a prerequisite is missing: the message names it and the recovery action |
| `DISABLED` | switched off in `CONFIG` (expensive stages are off by default) |

**Safety:** default mode `RESUME`; GPU training, DEM tile downloads, full inference and submission export are off until enabled below. Nothing is ever deleted on Drive, nothing is uploaded to the competition, no URL or token is written to disk. Pseudo-discovery scores are local-protocol numbers, never leaderboard scores.

Guide: `docs/MASTER_COLAB_GUIDE.md` · dependencies: `docs/PIPELINE_DEPENDENCIES.md` · recovery: `docs/ARTIFACT_RECOVERY.md`.

## 1. Configuration (control panel)

Edit, then run the cells top to bottom (or use the one-shot run in section 6). Keys are validated: a typo raises an error naming the closest valid key.

**Modes:** `SETUP_ONLY` (verify environment and existing data, no downloads) · `PREPARE_DATA` (download / preprocess missing data, EDA repair, splits, features) · `BASELINES` (+ restore / train baselines and the scoreboard) · `ADVANCED_RESEARCH` (+ PU, geometry, GeoStructureNet experiments) · `FULL_PIPELINE` (every enabled stage) · `RESUME` (continue from saved state; default) · `SUBMISSION_ONLY` (promoted model → GeoTIFF; prerequisites validated, not rebuilt).

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                                   # pin a commit SHA for reproducible experiments
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'run_mode': 'RESUME',
    'download_missing': True,                        # only files absent / invalid on Drive AND locally
    'rebuild_missing': True,

    'run_eda': True,                                 # minimal Phase-1 rebuild if outputs/eda is missing (CPU)
    'run_baselines': True,                           # scan / verify baselines (cheap)
    'train_baselines': [],                           # e.g. ['B0_lineament', 'G1', 'G3']; GPU: 'CNN_A_dti', 'DL00_CNN_B_dti', 'P4_B_gated'
    'run_advanced_features': True,
    'run_advanced_models': False,                    # GPU research experiments (opt-in)
    'advanced_experiments': ['b0_reference', 'b1_bg_weight', 'b2_nnpu_p010', 'c1_geometry_heads', 'e1_gsn_parity'],
    'run_dem_pilot': False,
    'run_scoreboard': True,
    'run_ensemble': True,
    'run_full_inference': False,                     # needs a PROMOTED model (manual decision)
    'run_submission': False,                         # writes GeoTIFFs; never uploads

    'use_gpu': True,
    'max_stage_minutes': 0,                          # e.g. 200 on free tier: clean, resumable stop
    'save_figures': True,
    'save_intermediate': True,
    'seed': 2026,
}

DEM_CONFIG = {
    'enabled': False,
    'mode': 'pilot',                                 # 'pilot' | 'inventory_only' (never the full collection)
    'download': False,                               # explicit approval, after reading the storage estimate
    'pilot_tiles': 24,                               # spatially balanced pilot (f1_dem_pilot.yaml)
    'max_tiles': 30,
    'max_gib': 15.0,
    'download_only_intersecting': True,
    'compute_morphology': True,                      # 1 m morphology before aggregation
    'aggregate_to_100m': True,
}

## 2–4. Google Drive, GitHub installation, package import

Idempotent: mounts Drive, clones (or updates) the repository at `CONFIG['ref']`, installs `gems-fault-discovery` in editable mode. A private repository is cloned with a token from Colab *Secrets* (`GITHUB_TOKEN`); the token is never printed or stored. PyTorch is **not** reinstalled (Colab's CUDA build is the right one).

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f"https://{token}@github.com/{CONFIG['repo']}.git" if token else f"https://github.com/{CONFIG['repo']}.git"
if not os.path.exists(f'{CODE}/.git'):
    subprocess.run(['git', 'clone', '-q', url, CODE], check=True)
subprocess.run(['git', '-C', CODE, 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', CODE, 'checkout', '-q', CONFIG['ref']], check=True)
# a branch ref follows origin (fast-forward only); a commit SHA has no origin/<sha> and stays pinned
subprocess.run(['git', '-C', CODE, 'merge', '-q', '--ff-only', 'origin/' + CONFIG['ref']], check=False, capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)                          # editable install is not visible to a running kernel
del token, url
print('commit', subprocess.run(['git', '-C', CODE, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

### Fresh download URLs (only when a raw file is missing on Drive)

DrivenData links are personal and **expire**. Preferred: Colab *Secrets* (key icon, enable notebook access) named `GEMS_URL_FEATURES`, `GEMS_URL_FAULTS`, `GEMS_URL_SUBMISSION` (and optionally `GEMS_URL_DEM_JSON`). Alternative: set `ASK_FOR_URLS = True` to paste them here (hidden input). URLs live only in memory; the acquisition manifest stores only the host and a hash. Files already valid on Drive are never downloaded again.

In [ ]:
URLS = {}
ASK_FOR_URLS = False
if ASK_FOR_URLS:
    from getpass import getpass
    for key in ('features', 'faults', 'submission', 'dem_json'):
        u = getpass(f'URL for {key} (Enter to skip): ').strip()
        if u:
            URLS[key] = u

from IPython.display import display
from gems_bench.pipeline import Pipeline
pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG, urls=URLS)
display(pipe.cfg.table())

## 5. Environment and GPU verification

`torch.cuda.is_available()` is what counts (a CPU-only PyTorch is not GPU support). GPU stages stop with *Runtime → Change runtime type → GPU* when no GPU is attached. Batch size, precision (bf16 only on compute capability ≥ 8) and inference batch adapt to the GPU found; the effective batch, patch size and epochs never change.

In [ ]:
pipe.step('bootstrap')
from gems_bench.training import gpu
g = pipe.pc.gpu()
print(g)
if CONFIG['use_gpu'] and not g.get('cuda'):
    print('No CUDA GPU in this runtime: GPU stages will stop with instructions. '
          'Runtime -> Change runtime type -> GPU, or set use_gpu=False.')
else:
    print('autoconfig (256 px, effective batch 16, 29 channels):',
          gpu.autoconfig(g, patch_px=256, effective_batch=16, n_channels=29))
display(__import__('pandas').read_csv('/content/gems_work/reports/pipeline/environment_report.csv'))

## 6. Persistent storage overview and execution plan

Read-only: validates what already exists **on Drive** and predicts each stage's action for the selected mode. Nothing is built here.

**One-shot alternative:** `pipe.run()` executes the whole configured mode in dependency order (the cells below run the same stages one section at a time, with displays).

In [ ]:
display(pipe.graph())
plan = pipe.plan()
display(plan)
from gems_bench.pipeline.status import storage
display(storage(pipe.pc))

RUN_EVERYTHING_NOW = False                           # True = run the configured mode in one call
if RUN_EVERYTHING_NOW:
    display(pipe.run())

## 7. Raw data inventory and one-time download

Per file: Drive (canonical path, then accepted alternates such as `data/raw/example_submission.tif`) → local disk → resumable download (`.partial`, HTTP Range, retries) → GeoTIFF / JSON validation → verified atomic copy to Drive → manifest. Downloads happen only for files that are missing or invalid in both places.

In [ ]:
from gems_bench.data.acquisition import inventory
display(inventory({'drive': pipe.pc.drive, 'local': pipe.pc.local}))
pipe.step('raw_data')

## 8. USGS 1 m DEM inventory (metadata only) and optional pilot

The inventory parses the URL JSON (`data/raw/tnm_items.json` is accepted and copied to `data/raw/dem_1m/tnm_urls.json`), identifies source projects, duplicate URLs, overlapping projects per tile, study-area intersection and the storage estimate, and writes a footprint GeoPackage and figure. **No tile is downloaded here.** The pilot (after the feature store, section 12) selects a spatially balanced subset and downloads it only with `DEM_CONFIG['download'] = True`.

In [ ]:
pipe.step('dem_inventory')
import json, pathlib
est = pathlib.Path('/content/gems_work/data/raw/dem_1m/inventory/dem_storage_estimate.json')
if est.exists():
    print(json.dumps(json.loads(est.read_text()), indent=1))

## 9. Native geospatial data verification

100 m GeoDAWN grid (EPSG:32611), mapped-fault labels (value ≥ 1), valid footprint, submission template on the same grid. Recorded in `environment/pipeline/native_grid.json` with raw-file fingerprints, so a new data release is detected instead of silently mixed.

In [ ]:
pipe.step('native_grid')
display(pipe.pc.data.summary())

## 10. Phase 1 EDA dependency repair

Only the EDA artifacts later phases actually read are regenerated (fault segments + archetypes, fused lineament maps, cross-modal orientation), with the same `gems_eda` functions and defaults as notebook 01, into `outputs/eda_rebuild/<version>/`. Each artifact group is installed into `outputs/eda/` **only if the whole group is absent** (no mixing with, or overwriting of, historical files) and listed in `manifest.json` with `origin: rebuilt`. Then the native-grid segment rasters are derived.

In [ ]:
pipe.step('geological_eda')
pipe.step('segment_attributes')
p = pathlib.Path('/content/gems_work/outputs/eda/manifest.json')
if p.exists():
    display(__import__('pandas').DataFrame(json.loads(p.read_text())).T)

## 11. Phase 2 geographic folds and pseudo-discovery splits

Protocol: 3 geographic folds, 20 km blocks, 13 km buffer · 3 discovery splits, 1 km fault-system linking · seed 2026. Existing complete folders are reused and validated (grid + parameters); they are never regenerated because an experiment starts. A changed protocol produces a new, differently named folder.

In [ ]:
pipe.step('spatial_folds')
pipe.step('discovery_splits')
print(pipe.pc.fold_rel, '|', pipe.pc.discovery_rel)
from gems_bench.pipeline.status import fold_figure, save_fig
fig = fold_figure(pipe.pc)
if fig is not None:
    display(fig); save_fig(pipe.pc, fig, 'validation_design')

## 12. Feature store: create once, reuse

Families: raw 19-channel geophysics · Phase 2 engineered · Phase 1 `eda:*` maps · Phase 4 remote-sensing structural features (those the baselines use, or all) · Phase 5 fault-normal profiles · 1 m DEM aggregates. Only missing features are computed; each family is pushed to Drive as soon as it exists. `data/feature_store/pipeline_feature_registry.csv` declares name, source, transform, scale, support, CRS / grid, NoData behaviour, standardisation and a spec hash per feature.

In [ ]:
pipe.step('feature_store')
reg = pathlib.Path('/content/gems_work/data/feature_store/pipeline_feature_registry.csv')
if reg.exists():
    fr = __import__('pandas').read_csv(reg)
    display(fr.groupby('family').agg(n=('name', 'size'), cached=('cached', 'sum')))

pipe.step('dem_pilot')                               # only runs when the DEM pilot is enabled

## 13. Baselines: restore or train

G1 terrain LightGBM, G3 engineered LightGBM, RS05 (Phase 4), CNN-A and CNN-B (distance-aware), gated dual encoder, plus the label-free B0 lineament floor. A reported score is not a model: each baseline is `AVAILABLE` only if its finished run folder and maps are found and verified. Otherwise it is `NOT_AVAILABLE`, or it is retrained from the committed configuration snapshot when listed in `train_baselines`. Regenerated runs are labelled `historical_equivalence: false`.

In [ ]:
pipe.step('baselines')
display(pipe.baselines())

## 14–15. Research experiments: PU supervision, geometry-aware decoding, GeoStructureNet, multiscale structure

Each experiment has a hypothesis, a control, a fixed split version, resumable units (3 CV folds + 3 discovery splits), checkpoints synced to Drive at intervals, and the section-17 promotion gate. Order matters: `b0_reference` is the control of the others. GPU only; enable with `run_advanced_models=True`.

In [ ]:
display(pipe.experiments())
pipe.step('advanced_models')

## 16–17. Geographic and pseudo-discovery validation, model comparison and promotion

One protocol for every available model: geographic DTI (cross-fitted area budget) and pseudo-discovery DTI (withheld systems excluded / retained as FP) are reported **separately**; `public_lb` comes only from your manual `reports/leaderboard_log.csv`. Promotion to `promoted` is a manual, evidence-backed registry transition.

In [ ]:
pipe.step('discovery_validation')
sb = pathlib.Path('/content/gems_work/reports/pipeline/scoreboard_latest.csv')
if sb.exists():
    b = __import__('pandas').read_csv(sb)
    display(b[[c for c in ('model', 'geo_dti_area', 'geo_fold_min', 'disc_dti_excl', 'disc_dti_retained_fp',
                           'short_recall_w', 'public_lb') if c in b.columns]])
from gems_bench.artifacts import ModelRegistry
display(ModelRegistry(pipe.pc.local / 'model_registry').table())
# Manual promotion (only when the candidate's promotion_decisions say promote=True):
# from gems_bench.experiment.promote import promote
# promote('<model_name>', '<experiment_id>'); pipe.push_all()

## 18. Ensemble optimisation

Members (≥ 2 with OOF + every discovery map) enter as per-fold rank quantiles; geographic weights are chosen on other folds, discovery weights with each split's withheld systems removed. An ensemble must pass the same gate against its best member.

In [ ]:
pipe.step('ensemble')

## 19–20. Full-area inference and competition GeoTIFF

Uses the registry **champion** (status `promoted`): fold models averaged over the full footprint (chunked, Hann-blended tiles), area-budget export policy, one validated single-band float32 GeoTIFF per known-fault policy in `submissions/` with provenance JSON. **Uploading is manual**; record each upload in `reports/leaderboard_log.csv`.

In [ ]:
pipe.step('full_inference')
pipe.step('submission')
import pandas as pd
from gems_bench.evaluation.scoreboard import LEADERBOARD_COLUMNS
log = pipe.pc.local / 'reports' / 'leaderboard_log.csv'
if not log.exists():
    pd.DataFrame(columns=LEADERBOARD_COLUMNS).to_csv(log, index=False)
display(pd.read_csv(log))

## 21. Figures and research reporting

`reports/pipeline/` on Drive: `PIPELINE_REPORT.md`, `stage_status.csv`, `dashboard_summary.csv`, `baseline_status.csv`, `experiment_status.csv`, `scoreboard_latest.csv`, `figures/*.png|svg`. Per-experiment tables and figures stay in `experiments/<id>/`.

In [ ]:
pipe.step('reports')
pipe.dashboard()

## 22. Final artifact and manifest validation

Re-validates every stage's outputs **on Drive** (what survives the session), lists the stage records and the latest events.

In [ ]:
pipe.push_all()
final = pipe.plan(mode='FULL_PIPELINE', root='drive')
display(final[['stage', 'outputs', 'planned', 'last_status', 'last_origin', 'last_finished', 'detail']])
display(pipe.status())
display(pipe.events(30))
for s in ('feature_store', 'baselines', 'discovery_validation', 'submission'):
    print(s, '->', pipe.outputs(s)[:3])

## 23. Resume instructions

After a disconnect or in a new session: **run sections 1–4 again, then the same cells (or `pipe.run()`)**.

* Finished stages are `SKIP_CACHED` (validated on Drive, staged back to `/content/gems_work` only when needed).
* Interrupted feature builds continue with the missing features; interrupted experiments continue at the next CV fold / discovery split and resume training from the last checkpoint pushed to Drive (`ckpt_sync_min`).
* `pipe.status()` shows the last state of every stage; `pipe.record('<stage>')` the full record (error, traceback tail, recovery action, code commit, fingerprints).
* To change a protocol (folds, features, experiment settings) edit the configuration: a new versioned folder / experiment id is created; earlier results are kept.
* On short sessions set `max_stage_minutes` (e.g. 200): training stops cleanly at an epoch boundary and resumes next time.

The original notebooks `00`–`07` remain as focused research entry points; this notebook is the recommended start.